# 🤖 Project 3 — AI Recommendation Logic
## Tech Stack Recommender | DecodeLabs Batch 2026
**Dataset:** Glassdoor Data Science Jobs (Kaggle) — 956 real job postings  
**Algorithm:** TF-IDF + Cosine Similarity (Content-Based Filtering)


In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("✅ All libraries imported!")

---
## 📂 Step 1 — Load the Kaggle Dataset
> **Source:** Glassdoor Data Science Jobs — available on Kaggle  
> Real job postings with titles, descriptions, salaries, locations.


In [ ]:
df = pd.read_csv("glassdoor_jobs.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

---
## 🔍 Step 2 — Explore the Data


In [ ]:
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total job postings : {len(df)}")
print(f"Unique job titles  : {df['Job Title'].nunique()}")
print(f"Industries covered : {df['Industry'].nunique()}")
print()
print("Top 10 Job Titles:")
print(df['Job Title'].value_counts().head(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#7F77DD","#5DCAA5","#D85A30","#378ADD","#639922","#BA7517","#D4537E","#888780","#E24B4A","#1D9E75"]

top_titles = df["Job Title"].value_counts().head(10)
axes[0].barh(top_titles.index[::-1], top_titles.values[::-1], color=colors)
axes[0].set_xlabel("Count")
axes[0].set_title("Top 10 Job Titles in Dataset", fontsize=13, fontweight="bold")
axes[0].tick_params(axis="y", labelsize=9)

sector_counts = df["Sector"].value_counts().head(6)
axes[1].pie(sector_counts.values, labels=sector_counts.index,
            autopct="%1.1f%%", colors=colors[:6], startangle=140)
axes[1].set_title("Job Distribution by Sector", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

---
## ⚙️ Step 3 — Feature Extraction: Skills from Job Descriptions


In [ ]:
TECH_SKILLS = [
    "python","r","java","scala","sql","julia","c++","javascript","typescript","go","bash","matlab",
    "machine learning","deep learning","neural network","nlp","computer vision",
    "tensorflow","pytorch","keras","scikit-learn","xgboost","lightgbm",
    "pandas","numpy","spark","hadoop","kafka","airflow","dbt",
    "tableau","power bi","excel","looker",
    "aws","azure","gcp","docker","kubernetes","terraform","ci/cd","git","linux","jenkins",
    "postgresql","mysql","mongodb","redis","snowflake","redshift","elasticsearch","oracle",
    "statistics","probability","a/b testing","data visualization",
    "etl","data mining","feature engineering","regression","classification",
    "clustering","time series","api","rest","agile"
]

def extract_skills(text):
    if not isinstance(text, str): return []
    text_lower = text.lower()
    return list(set([s for s in TECH_SKILLS if re.search(r"\b" + re.escape(s) + r"\b", text_lower)]))

df["extracted_skills"] = df["Job Description"].apply(extract_skills)
df["skill_count"]      = df["extracted_skills"].apply(len)
df["skills_str"]       = df["extracted_skills"].apply(lambda x: " ".join(x))

print(f"Skills extracted from {len(df)} postings")
print(f"Average skills per posting: {df['skill_count'].mean():.1f}")
print(f"Postings with 0 skills: {(df['skill_count']==0).sum()}")
df_clean = df[df["skill_count"] > 0].reset_index(drop=True)
print(f"\nClean dataset: {len(df_clean)} postings")

---
## 📊 Step 4 — Skills Analysis


In [ ]:
all_skills = [s for skills in df_clean["extracted_skills"] for s in skills]
skill_freq = Counter(all_skills).most_common(20)
skill_names  = [s[0] for s in skill_freq]
skill_counts = [s[1] for s in skill_freq]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

bar_colors = ["#7F77DD" if i<5 else "#AFA9EC" if i<10 else "#CECBF6" for i in range(20)]
axes[0].barh(skill_names[::-1], skill_counts[::-1], color=bar_colors[::-1])
axes[0].set_title("Top 20 Most In-Demand Skills (Kaggle Dataset)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Frequency in Job Postings")
legend_elements = [mpatches.Patch(color="#7F77DD",label="Top 5"),
                   mpatches.Patch(color="#AFA9EC",label="Top 10"),
                   mpatches.Patch(color="#CECBF6",label="Top 20")]
axes[0].legend(handles=legend_elements)

top5 = df_clean["Job Title"].value_counts().head(5).index
data_box = [df_clean[df_clean["Job Title"]==t]["skill_count"].values for t in top5]
bp = axes[1].boxplot(data_box, patch_artist=True, labels=[t.replace(" ","\n") for t in top5])
for patch, c in zip(bp["boxes"],["#E1F5EE","#EEEDFE","#FAECE7","#E6F1FB","#EAF3DE"]):
    patch.set_facecolor(c)
axes[1].set_title("Skills Count per Job Title", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Number of Skills")

plt.tight_layout()
plt.show()

---
## 🧮 Step 5 — TF-IDF Vectorization
**TF** = how often a skill appears in one posting  
**IDF** = log(total postings / postings with that skill) — penalizes very common skills  
**TF-IDF** = TF × IDF — rare specific skills get higher weights


In [ ]:
tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(df_clean["skills_str"])
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} job postings (documents)")
print(f"  → {tfidf_matrix.shape[1]} unique skill terms in vocabulary")
print(f"\nSample vocabulary: {tfidf.get_feature_names_out()[:15].tolist()}")

---
## 📐 Step 6 — Cosine Similarity + Recommendation Engine


In [ ]:
def recommend_jobs(user_skills, top_n=5):
    """
    4-Step Pipeline:
    1. Ingestion  - accept user skill list
    2. Scoring    - TF-IDF transform + cosine similarity vs all postings
    3. Sorting    - rank by descending similarity score
    4. Filtering  - return top-N deduplicated job titles
    """
    user_input = " ".join([s.lower().strip() for s in user_skills])
    user_vec   = tfidf.transform([user_input])
    sim_scores = cosine_similarity(user_vec, tfidf_matrix).flatten()
    ranked_idx = sim_scores.argsort()[::-1]

    seen, results = set(), []
    for idx in ranked_idx:
        title = df_clean.iloc[idx]["Job Title"]
        if title not in seen:
            seen.add(title)
            matched = [s for s in user_skills if s.lower() in df_clean.iloc[idx]["skills_str"]]
            results.append({
                "rank":           len(results)+1,
                "job_title":      title,
                "company":        df_clean.iloc[idx]["Company Name"],
                "location":       df_clean.iloc[idx]["Location"],
                "salary":         df_clean.iloc[idx]["Salary Estimate"],
                "industry":       df_clean.iloc[idx]["Industry"],
                "match_score":    round(sim_scores[idx]*100, 2),
                "matched_skills": matched,
                "role_skills":    df_clean.iloc[idx]["extracted_skills"]
            })
        if len(results) >= top_n:
            break
    return pd.DataFrame(results)

print("✅ Recommendation engine ready!")

---
## 🎯 Step 7 — Run Recommendations


In [ ]:
medals = {1:"🥇",2:"🥈",3:"🥉",4:"4️⃣",5:"5️⃣"}

user_skills_1 = ["python","machine learning","sql","tensorflow","statistics"]
print("=" * 60)
print(f"USER SKILLS: {user_skills_1}")
print("=" * 60)
results_1 = recommend_jobs(user_skills_1, top_n=5)
for _, row in results_1.iterrows():
    print(f"\n{medals[row['rank']]} #{row['rank']} {row['job_title']}")
    print(f"   Company  : {row['company']}")
    print(f"   Location : {row['location']}")
    print(f"   Salary   : {row['salary']}")
    print(f"   Match    : {row['match_score']}%")
    print(f"   Matched  : {row['matched_skills']}")

In [ ]:
user_skills_2 = ["aws","docker","kubernetes","linux","python","ci/cd"]
print("=" * 60)
print(f"USER SKILLS: {user_skills_2}")
print("=" * 60)
results_2 = recommend_jobs(user_skills_2, top_n=5)
for _, row in results_2.iterrows():
    print(f"\n{medals[row['rank']]} #{row['rank']} {row['job_title']}  —  {row['match_score']}% match")
    print(f"   Location: {row['location']} | Salary: {row['salary']}")

In [ ]:
user_skills_3 = ["sql","tableau","power bi","excel","statistics","data visualization"]
print("=" * 60)
print(f"USER SKILLS: {user_skills_3}")
print("=" * 60)
results_3 = recommend_jobs(user_skills_3, top_n=5)
for _, row in results_3.iterrows():
    print(f"\n{medals[row['rank']]} #{row['rank']} {row['job_title']}  —  {row['match_score']}% match")
    print(f"   Location: {row['location']} | Salary: {row['salary']}")

---
## 📈 Step 8 — Visualize Results


In [ ]:
def plot_recommendations(results, title="Top Job Matches"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors_bar = ["#7F77DD","#5DCAA5","#D85A30","#378ADD","#BA7517"]

    bars = axes[0].barh(results["job_title"][::-1], results["match_score"][::-1],
                        color=colors_bar[:len(results)][::-1])
    axes[0].set_xlabel("Cosine Similarity Score (%)")
    axes[0].set_title(title, fontsize=12, fontweight="bold")
    axes[0].set_xlim(0, max(results["match_score"]) * 1.25)
    for bar, score in zip(bars, results["match_score"][::-1]):
        axes[0].text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
                     f"{score}%", va="center", fontsize=10)

    match_counts = results["matched_skills"].apply(len)
    axes[1].bar(results["job_title"], match_counts, color=colors_bar[:len(results)])
    axes[1].set_ylabel("Skills Matched")
    axes[1].set_title("Matched Skills per Role", fontsize=12, fontweight="bold")
    axes[1].tick_params(axis="x", rotation=25)
    for i, v in enumerate(match_counts):
        axes[1].text(i, v+0.05, str(v), ha="center", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

plot_recommendations(results_1, "Data Science Profile — Top 5 Matches")

---
## 🖥️ Step 9 — Interactive Section (Enter YOUR Skills)


In [ ]:
# ── CHANGE THESE TO YOUR OWN SKILLS ──
MY_SKILLS = [
    "python",
    "machine learning",
    "sql",
    "deep learning",
    "pytorch",
    "statistics",
]

print("Searching for your best job matches...\n")
my_results = recommend_jobs(MY_SKILLS, top_n=5)

print("┌" + "─"*58 + "┐")
print("│" + " 🎯  YOUR TOP JOB RECOMMENDATIONS ".center(58) + "│")
print("└" + "─"*58 + "┘")

for i, row in my_results.iterrows():
    bar = "█" * int(row["match_score"]/5) + "░" * (20-int(row["match_score"]/5))
    print(f"\n  {medals[row['rank']]} #{row['rank']} {row['job_title']}")
    print(f"     Match  : [{bar}] {row['match_score']}%")
    print(f"     Salary : {row['salary']}")
    print(f"     ✔ Matched: {', '.join(row['matched_skills']) if row['matched_skills'] else 'None directly'}")

plot_recommendations(my_results, "My Skills — Top 5 Matches")

---
## 🔬 Step 10 — Algorithm Deep Dive


In [ ]:
sample_skills = ["python","machine learning","sql"]
user_vec_demo = tfidf.transform([" ".join(sample_skills)])
vocab = tfidf.get_feature_names_out()

non_zero = user_vec_demo.nonzero()[1]
print("📌 TF-IDF weights for user vector:")
for idx in non_zero[:10]:
    print(f"   {vocab[idx]:<25} weight = {user_vec_demo[0, idx]:.4f}")

all_sims = cosine_similarity(user_vec_demo, tfidf_matrix).flatten()
print(f"\n📊 Similarity score distribution:")
print(f"   Max  : {all_sims.max():.4f}  ← Best match")
print(f"   Mean : {all_sims.mean():.4f}")
print(f"   Min  : {all_sims.min():.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_sims, bins=40, color="#7F77DD", edgecolor="white", linewidth=0.5)
ax.axvline(all_sims.max(), color="#D85A30", linestyle="--", linewidth=2,
           label=f"Best match score: {all_sims.max():.3f}")
ax.set_xlabel("Cosine Similarity Score")
ax.set_ylabel("Number of Job Postings")
ax.set_title("Distribution of Similarity Scores (python + ML + SQL)", fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

---
## ✅ Project Summary

| Step | Stage | What Happens |
|------|-------|-------------|
| 1 | Load | Glassdoor Kaggle dataset (956 real job postings) |
| 2 | Clean | Extract 50+ tech skills from raw descriptions via regex |
| 3 | EDA | Visualize skill frequency, job distribution |
| 4 | TF-IDF | Convert skill lists to weighted numerical vectors |
| 5 | Cosine Sim | Measure user ↔ job alignment (angle, not distance) |
| 6 | Rank | Sort all postings by descending score |
| 7 | Filter | Return Top-N unique job titles |

**Project 3 Complete ✅ — DecodeLabs Batch 2026** 🎓
